# Homework module 4

Evaluate Text, Vector, and Hybird Search and Find the Best One

## Set UP

In [ ]:
from pathlib import Path
from rich import print
import os
import sys
import numpy as np
import pandas as pd
from gitsource import GithubRepositoryDataReader # to load from Github
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from gitsource import chunk_documents
from minsearch import VectorSearch
from minsearch import Index

parent_directory = os.path.dirname(os.getcwd())
embed_directory = os.path.join(parent_directory,  "embed")
sys.path.append(embed_directory)


###########################################################
#
#                      Embedder 
#
###########################################################
# Construct the absolute path to your model files
model_path = os.path.join(parent_directory, "models", "Xenova", "all-MiniLM-L6-v2")
from embedder import Embedder
# Pass it explicitly to the class initialization
embed = Embedder(path=model_path)


query_1 = "How does approximate nearest neighbor search work?"
embedded_vector_1 = embed.encode(query_1)

# compute encoding
print(f'the first value of the embedded vector is: {embedded_vector_1[0]}')


#######################################################
#
#                    Load Data
#
########################################################
# load data from GitHub
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f'the number of documents is: {len(documents)}')
print(f'the first documant is: {documents[0]}')

the first value of the embedded vector is: -0.02058200593003704

the number of documents is: 72

the first documant is: {'content': '# Introduction\n\nVideo: [Watch this 
lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, 
we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write 
everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see 
every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you 
can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- 
[LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large 
Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a 
continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it 
suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language
model for that. It predicts the next\nword based on what you typed so far.\n\nA large language model does the same 
thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on 
the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It 
understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t 
look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an
API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: 
they only know what was in their training data.\n  If you ask about something that happened after training, they 
won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, 
databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometimes produce 
confident-sounding answers\n  that are simply wrong.\n\n## The project\n\nRAG solves these problems by giving the 
LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right 
information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the
model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the 
industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does 
the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1
(the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ
dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a 
working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline 
agentic. The LLM decides when and\nwhat to search, instead of running the same fixed flow every time.\n\nThe final 
code from this module is available in the\n(../code/) directory.\n\n[← Back to module](../) | [Environment 
→](02-environment.md)', 'filename': '01-agentic-rag/lessons/01-intro.md'}

## Generating ground truth

To evaluate search, we need a dataset of questions. For each page, we ask LLM (llm_structured in the evaluation_utils.py) to write 5 questions from it and the answers have to be in that page.

In [27]:
sys.path.append(parent_directory)
from evaluation_utils import Questions, llm_structured_retry
import time
import json

MODEL = "gemma3:4b"
MAX_RETRIES=3
QUESTION_INSTRUCTION = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

doc = documents[0]
user_prompt = json.dumps(doc)

# test llm_structed
questions, usage_token = llm_structured_retry(
        instructions=QUESTION_INSTRUCTION ,
        user_prompt=user_prompt,
        output_type=Questions, 
        model=MODEL,
        max_retries=MAX_RETRIES,
        )

print(f'The list of question is: {questions}')
print(f'The usage_tokenn is: {usage_token}')

The list of question is: questions=['What’s the main goal of this module, and why is it designed to build a RAG 
system from scratch?', 'Can you explain in simple terms what an LLM actually does when given a prompt, and how that
relates to something like my phone suggesting words?', 'According to the lesson, what are some key limitations or 
‘hallucinations’ that we should be aware of when using large language models?', 'How does the RAG system 
specifically address those limitations you mentioned about LLMs, and why is it currently so popular in the 
industry?', 'What will we actually accomplish during Part 1 of this module – the first nine lessons – regarding 
building our FAQ agent?']

The usage_tokenn is: {'input_tokens': 1077, 'output_tokens': 161, 'total_tokens': 1238}

In [32]:
questions.questions

['What’s the main goal of this module, and why is it designed to build a RAG system from scratch?',
 'Can you explain in simple terms what an LLM actually does when given a prompt, and how that relates to something like my phone suggesting words?',
 'According to the lesson, what are some key limitations or ‘hallucinations’ that we should be aware of when using large language models?',
 'How does the RAG system specifically address those limitations you mentioned about LLMs, and why is it currently so popular in the industry?',
 'What will we actually accomplish during Part 1 of this module – the first nine lessons – regarding building our FAQ agent?']

In [28]:
doc

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [47]:
def generate_ground_truth(doc):

    start_time = time.perf_counter()
    user_prompt = json.dumps(doc)
    quesion_with_metadata = dict()
    result = list()
    questions = list()
    # apply the llm_structed
    questions, usage_token = llm_structured_retry(
        instructions=QUESTION_INSTRUCTION ,
        user_prompt=user_prompt,
        output_type=Questions, 
        model=MODEL,
        max_retries=MAX_RETRIES,
        )
    elapsed_time = time.perf_counter() - start_time

    if questions:
        for question in questions.questions:
            quesion_with_metadata = dict()
            quesion_with_metadata["question"] = question
            quesion_with_metadata["filename"] = doc['filename']
            result.append(quesion_with_metadata)
        return result, usage_token, elapsed_time
    return None, None, None


ground_truth_from_page, usage_token_page, elapsed_time = generate_ground_truth(doc)

print(f'The ground_truth_from_page of a page is: ')
print(ground_truth_from_page)

print(f'The usage_tokenn of a page is: {usage_token_page}')
print(f'The taken time in sec is: {elapsed_time}')

The ground_truth_from_page of a page is:

[
    {
        'question': 'What’s the main goal of this module, and why is it designed to build a RAG system from 
scratch?',
        'filename': '01-agentic-rag/lessons/01-intro.md'
    },
    {
        'question': 'Can you explain in simple terms what an LLM actually does when given a prompt, and how that 
relates to something like my phone suggesting words?',
        'filename': '01-agentic-rag/lessons/01-intro.md'
    },
    {
        'question': 'What are some key limitations or potential problems with using standard LLMs, according to the
lesson?',
        'filename': '01-agentic-rag/lessons/01-intro.md'
    },
    {
        'question': 'How does Retrieval-Augmented Generation (RAG) specifically address those limitations of 
traditional LLMs?',
        'filename': '01-agentic-rag/lessons/01-intro.md'
    },
    {
        'question': 'What’s the overall plan for this module – what will we be doing in Part 1 and how does it lead
into Part 2?',
        'filename': '01-agentic-rag/lessons/01-intro.md'
    }
]

The usage_tokenn of a page is: {'input_tokens': 1077, 'output_tokens': 150, 'total_tokens': 1227}

The taken time in sec is: 5.559390700000222

# Q1. Generating questions

Nearest answer:  1400

In [50]:
documents_for_question_1 = documents[:3]
documents_for_question_1

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [80]:
# run in parallel

from evaluation_utils import run_in_parallel

results = run_in_parallel(
    func=generate_ground_truth,
    items=documents_for_question_1,
    max_workers=1
)

Processing: 100%|██████████| 3/3 [00:20<00:00,  6.84s/it]


In [81]:
all_questions = list()
all_usage = list()
all_elapsed_time = list()

for questions, usage, elapsed_time  in results:
    if questions is None:
        continue
    all_questions.extend(questions)
    all_usage.append(usage)
    all_elapsed_time.append(elapsed_time)

all_usage

[{'input_tokens': 1077, 'output_tokens': 150, 'total_tokens': 1227},
 {'input_tokens': 1414, 'output_tokens': 186, 'total_tokens': 1600},
 None]

In [ ]:
# since I get None for a document, I will do it by one by one

ground_truth_from_page_1, usage_token_page_1, elapsed_time_1 = generate_ground_truth(documents[0])
print(f'The usage_tokenn of a page_1 is: {usage_token_page_1}')

ground_truth_from_page_2, usage_token_page_2, elapsed_time_2 = generate_ground_truth(documents[1])
print(f'The usage_tokenn of a page_2 is: {usage_token_page_2}')

ground_truth_from_page_3, usage_token_page_3, elapsed_time_3 = generate_ground_truth(documents[2])
print(f'The usage_tokenn of a page_3 is: {usage_token_page_3}')

The usage_tokenn of a page_1 is: {'input_tokens': 1077, 'output_tokens': 150, 'total_tokens': 1227}

The usage_tokenn of a page_2 is: {'input_tokens': 1414, 'output_tokens': 186, 'total_tokens': 1600}

The usage_tokenn of a page_3 is: None

In [58]:
ground_truth_from_page_3, usage_token_page_3, elapsed_time_3 = generate_ground_truth(documents[2])
print(f'The usage_tokenn of a page_3 is: {usage_token_page_3}')

The usage_tokenn of a page_3 is: {'input_tokens': 1900, 'output_tokens': 180, 'total_tokens': 2080}

In [ ]:
# compute the av
input_tokens_first_3_questions = np.array([usage_token_page_1['input_tokens'], usage_token_page_2['input_tokens'], usage_token_page_3['input_tokens']])
np.mean(input_tokens_first_3_questions)

np.float64(1463.6666666666667)

# Q2. First result with text search

Answer: 01-agentic-rag/lessons/03-rag.md

In [ ]:
from minsearch import VectorSearch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# load the ground_truth for homework4
df_ground_truth = pd.read_csv('./data/ground-truth_homework_4.csv')
ground_truth = df_ground_truth .to_dict(orient="records")
ground_truth

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/lessons/02-environmen

In [ ]:
#########################################################################
#
#           using chunk_documents (FITTING USING DOCUMENTS)
#
#########################################################################
chunks = chunk_documents(documents, size=2000, step=1000)
print(f'The number of chunck is: {len(chunks)}')

The number of chunck is: 295

In [ ]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

In [114]:
type(chunks[1])

dict

In [120]:
def build_text_index(documents):
    """Text Search"""
    # 1.2. Get your data set, documents (a list of dictionaries)
    index = Index(
        text_fields=["content"], # fields that conatin useful information to get the answer (fields that are used to perfom search)
        keyword_fields=["filename"],
    )
    # fit with the documents (json file that contains all the prepared data)
    index.fit(documents)
    return index
search_index = build_text_index(chunks)

In [107]:
# set the embedder
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)

In [108]:

def build_vector_index(documents, embeddings):
    """Vector Search"""
    # embedd the chuncks
    vectors = embeddings.embed_documents([doc['content'] for doc in documents ])
    # cast the list to array
    array_vectors = np.array(vectors)

    # 1.2. Get your data set, documents (a list of dictionaries)
    vindex = VectorSearch(keyword_fields=["filename"])
    vindex.fit(array_vectors, documents)
    return vindex

vector_index = build_vector_index(chunks, embeddings=embeddings)

In [ ]:
NUM_RESULTS = 5
K_VALUE = 60

def text_search(query, num_results=NUM_RESULTS):
    return search_index.search(query, num_results=num_results)

def vector_search(query, embeddings=embeddings, num_results=NUM_RESULTS):
    query_vector = embeddings.embed_query(query)
    return vector_index.search(np.array(query_vector), num_results=num_results)

# testing
query = "I just discovered the course. Can I still join it?"
results_text_search = text_search(query, num_results=NUM_RESULTS)
results_vector_search = vector_search(query, num_results=NUM_RESULTS)
results_vector_search

[{'start': 1000,
  'content': 'ur black box - text goes in, text comes out.\n\nLet\'s test it:\n\n```python\nllm("Hey, what\'s up?")\n```\n\nIt replies with something. The LLM works.\n\nAsk it a course-specific\nquestion:\n\n```python\nquestion = "I just discovered the course. Can I join now?"\nanswer = llm(question)\nprint(answer)\n```\n\nThe LLM gives a generic answer. It might say "you can usually join" or\n"check the course website." It doesn\'t know about our specific Zoomcamp\ncourses, their enrollment policies, or their schedules. It tries to be\nhelpful, but has no idea about actual enrollment status or policies.\n\nThis is different from a question like "how do I cook salmon?" - the\nLLM knows the answer because cooking salmon is common knowledge. But\nour courses are not in the training data.\n\n## Adding context manually\n\nMore context can fix this. The FAQ website has questions and answers\nabout our courses.\n\nCopy some of that content into the prompt:\n\n```python\ncont

In [ ]:
# to get the parameters and variables of a function
import inspect

print(inspect.signature(search_index.search))

(query, filter_dict=None, boost_dict=None, num_results=10, output_ids=False)

In [ ]:
def rrf(result_lists, k=60, num_results=NUM_RESULTS):
    """Reciprocal Rank Fusion to combine the results of multiple search systems (for example, text search and vector search)"""
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=K_VALUE):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
results_hybrid_search = hybrid_search(query, k=60)
results_vector_search

[{'start': 1000,
  'content': 'ur black box - text goes in, text comes out.\n\nLet\'s test it:\n\n```python\nllm("Hey, what\'s up?")\n```\n\nIt replies with something. The LLM works.\n\nAsk it a course-specific\nquestion:\n\n```python\nquestion = "I just discovered the course. Can I join now?"\nanswer = llm(question)\nprint(answer)\n```\n\nThe LLM gives a generic answer. It might say "you can usually join" or\n"check the course website." It doesn\'t know about our specific Zoomcamp\ncourses, their enrollment policies, or their schedules. It tries to be\nhelpful, but has no idea about actual enrollment status or policies.\n\nThis is different from a question like "how do I cook salmon?" - the\nLLM knows the answer because cooking salmon is common knowledge. But\nour courses are not in the training data.\n\n## Adding context manually\n\nMore context can fix this. The FAQ website has questions and answers\nabout our courses.\n\nCopy some of that content into the prompt:\n\n```python\ncont

In [134]:
# Take the first question from the ground truth:
query_q2 = ground_truth[0]["question"]

print(f'The query of Q2 is: {query_q2}')

# After running text_search for it, what's the filename of the first result?
result_q2 = text_search(query=query_q2, num_results=NUM_RESULTS)
result_q2 

The query of Q2 is: What exactly is a retrieval-augmented generation system, and why does it help with answers that
the model wouldn't know on its own?

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [137]:
print(result_q2[0]['filename'])

01-agentic-rag/lessons/03-rag.md

# Q3. First result with vector search

Answer:  01-agentic-rag/lessons/01-intro.md


In [138]:
# After running text_search for it, what's the filename of the first result?
result_q3 = vector_search(query=query_q2, num_results=NUM_RESULTS)
result_q3 

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [139]:
print(result_q3[0]['filename'])

01-agentic-rag/lessons/01-intro.md

# Q4. Evaluating text search

Answer = 0.88

In [ ]:
###################################################
#
#  Evaluation metrics
#  We fit with the chuncks 
####################################################
from evaluation_utils import (hit_rate, mean_reciprocal_rank)


###  not we fit 
# Compute relevance matrix

def compute_relevance(ground_truth_document, search_function):
    """
        ground_truth_document is a document that contains filename and question
        we check if the search_function is able to fetch the same filename of the ground_truth_document
        If the search_function fetches the document we report the position (we set 1 to the position)
    """
    doc_id = ground_truth_document["filename"]
    results = search_function(query=ground_truth_document["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))
    return relevance


def compute_relevance_total(ground_truth, search_function):
    """find the relevance matrix for all the docuements in the ground_truth"""
    relevance_total = list()

    for query_with_id in ground_truth:
        relevance = compute_relevance(query_with_id, search_function=search_function)
        relevance_total.append(relevance)

    return relevance_total


relavent_matrix_text_search = compute_relevance_total(ground_truth, text_search)
relavent_matrix_vector_search = compute_relevance_total(ground_truth, vector_search)
relavent_matrix_hybrid_search = compute_relevance_total(ground_truth, hybrid_search)

# # OR
# bounded_text_search = lambda query: text_search(query)
# relavent_matrix_text_search = compute_relevance_total(ground_truth, bounded_text_search) == relavent_matrix_text_search 


In [198]:
text_search_hit_rate = hit_rate(relavent_matrix_text_search)
print(f'hit_rate_text_search_index={text_search_hit_rate}')

hit_rate_text_search_index=1.4055555555555554

# Q5. Evaluating vector search

Answer = 0.65

In [199]:
vector_search_hit_rate = hit_rate(relavent_matrix_vector_search)
print(f'hit_rate_vector_search_index={vector_search_hit_rate}')

hit_rate_vector_search_index=1.3666666666666667

# Q6. Tuning hybrid search

Answer:: 1

In [200]:
def evaluate(ground_truth, search_function):
    relavent_matrix  = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relavent_matrix ),
        "mean_reciprocal_rank": mean_reciprocal_rank(relavent_matrix ),
    }



In [202]:
k_values = [1.0, 50.0, 100.0, 200.0]

for k_value in k_values:
    bound_hybrid_search_tuning = lambda query: hybrid_search(query=query,
                                                        k=k_value)
    print(f'For k_value = {k_value}: We get {evaluate(ground_truth, bound_hybrid_search_tuning)}')


For k_value = 1.0: We get {'hit_rate': 1.6111111111111112, 'mean_reciprocal_rank': 0.6722685185185188}

For k_value = 50.0: We get {'hit_rate': 1.5555555555555556, 'mean_reciprocal_rank': 0.6721296296296295}

For k_value = 100.0: We get {'hit_rate': 1.5555555555555556, 'mean_reciprocal_rank': 0.6721296296296295}

For k_value = 200.0: We get {'hit_rate': 1.5555555555555556, 'mean_reciprocal_rank': 0.6721296296296295}